# Surfaces, reclassification et transitions

**Cartomize · Version 1.0**

Durée indicative : 30 minutes.

## Objectifs

Produire une nomenclature binaire, calculer les superficies et distinguer pertes, gains et bilan net.

## Apport de Cartomize

Les rasters reclassés, les tableaux et la matrice de transition partagent la même grille et les mêmes règles de NoData.

Données : paysage pédagogique généré avec une graine fixe. Les cartes et mesures ne décrivent aucun site réel.

## Préparation

Installer l’environnement décrit dans le [guide de démarrage](../README.md), puis exécuter les cellules dans l’ordre. Chaque exécution utilise un nouveau dossier de résultats.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from IPython.display import display, Image
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'course.py').is_file() or (p/'tutorials/course.py').is_file())
root = root if (root/'course.py').is_file() else root/'tutorials'
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import cartomize as cm
from course import demo, run_directory, show_raster, show_rgb, gallery, CRS, CLASSES, ROLES

In [2]:
out = run_directory("05_surfaces_changements")
d = demo(out/"data")

## 1. Définir la nomenclature

Deux dates simulées utilisent les mêmes codes : végétation, sol et eau. Le code 255 est NoData. Le code 0 du résultat binaire reste une valeur valide.

In [3]:
before = cm.raster.reclassify(d['before'], out/'vegetation_t0.tif', {1:1,2:0,3:0})
after = cm.raster.reclassify(d['after'], out/'vegetation_t1.tif', {1:1,2:0,3:0})
areas0=cm.raster.class_areas(before,unit='ha')
areas1=cm.raster.class_areas(after,unit='ha')
display(areas0);display(areas1)

,class,pixels,area_ha
0,0.0,5103,459.27
1,1.0,3490,314.10


,class,pixels,area_ha
0,0.0,4718,424.62
1,1.0,3875,348.75


## 2. Quantifier les transitions

Une matrice de transition nécessite des grilles identiques. La surface d’un pixel de 30 m vaut 0,09 ha. Les gains et pertes sont calculés séparément avant leur différence.

In [4]:
matrix=cm.raster.change_matrix(before,after)
with rasterio.open(before) as src:
    pixel_ha=abs(src.transform.a*src.transform.e-src.transform.b*src.transform.d)/10000
matrix['surface_ha']=matrix['pixels']*pixel_ha
matrix.to_csv(out/'transitions.csv',index=False)
display(matrix)
loss=matrix.loc[(matrix['from']==1)&(matrix['to']==0),'surface_ha'].sum()
gain=matrix.loc[(matrix['from']==0)&(matrix['to']==1),'surface_ha'].sum()
pd.DataFrame({'indicateur':['Perte simulée','Gain simulé','Bilan net'],'surface_ha':[loss,gain,gain-loss]})

Out[4]: 
      indicateur  surface_ha
0  Perte simulée       12.87
1    Gain simulé       47.52
2      Bilan net       34.65


,from,to,pixels,surface_ha
0,0.0,0.0,4575,411.75
1,0.0,1.0,528,47.52
2,1.0,0.0,143,12.87
3,1.0,1.0,3347,301.23


In [5]:
change=cm.calculate('where((a==1)&(b==0),1,where((a==0)&(b==1),2,0))',{'a':before,'b':after},out/'changements.tif')
carte=cm.Map(title='Transitions de végétation',subtitle='Scénario pédagogique, deux dates simulées',crs=CRS,credits='Données synthétiques | 30 m')
carte.add_layer(change,name='Transition',classes={0:('Stable','#e8e8e3'),1:('Perte simulée','#a55b3e'),2:('Gain simulé','#418a61')})
carte.add_layer(d['zones'],name='Secteurs',color='none',edgecolor='#555555',legend=False)
carte.export(out/'transitions.pdf',dpi=300)
fig=carte.render(dpi=130);gallery(fig,'05_changements');plt.show()

## 3. Vérifier la conservation des surfaces

In [6]:
with rasterio.open(before) as a, rasterio.open(after) as b:
    aa=a.read(1,masked=True);bb=b.read(1,masked=True)
    common=~(np.ma.getmaskarray(aa)|np.ma.getmaskarray(bb))
    assert matrix['pixels'].sum()==common.sum()
    expected=(np.count_nonzero((bb.data==1)&common)-np.count_nonzero((aa.data==1)&common))*pixel_ha
    assert np.isclose(gain-loss,expected)

## 4. Extraire et reprojeter un raster catégoriel

Le voisin le plus proche évite de créer des codes intermédiaires. `raster.clip` et `raster.reproject` traitent une bande sélectionnée ; pour un multibande scientifique, utiliser le prétraitement multispectral.

In [7]:
clipped=cm.raster.clip(d['before'],cm.read_file(d['aoi']),out/'classes_decoupees.tif')
projected=cm.raster.reproject(clipped,out/'classes_geographiques.tif','EPSG:4326',resampling='nearest')
with rasterio.open(projected) as src:
    assert set(np.unique(src.read(1,masked=True).compressed())) <= {1,2,3}
    assert src.crs.to_epsg()==4326
print('Codes préservés après extraction et reprojection.')

Codes préservés après extraction et reprojection.


## Exercice

Pourquoi le composite de Mvouti couvrant 2024–2025 ne permet-il pas, à lui seul, de calculer un changement 2024 → 2025 ?

<details>
<summary>Éléments de correction</summary>

Il s’agit d’un seul produit composite. Il faut deux observations ou classifications datées séparément, comparables, alignées et validées. Découper ou dupliquer ce composite ne crée pas une seconde observation.

</details>

## Reproduction

Les paramètres, la graine et les chemins sont explicites dans les cellules. Le répertoire `out` contient les produits de cette exécution. Adapter les sources, le système de coordonnées et les paramètres avant de transférer la méthode à une étude.